# Analytics Hub - Upload Router

Picks up the CSV exports dropped into **`Files/analytics_hub_uploads/`** and hands each one to the
notebook that loads it. Nobody has to rename a file or choose a folder: the router recognises
each export from its **header row**.

| Export | Recognised by | Goes to | Loaded by |
|---|---|---|---|
| Product feedback (M365 admin center > Health > Product feedback) | Feedback Id, Date Submitted UTC, Feedback Type | `Files/product_feedback/` (kept, de-duplicated by Feedback Id) | `Copilot_ProductFeedback_Ingester` |
| Agent 365 agents (M365 admin center > Agents > All agents > Export) | an agent name plus Title ID / creator / type columns | `Files/agent365/agents.csv` (replaced) | `Copilot_Agent365_Lander` |
| Copilot Studio credits, 3 exports (Power Platform admin center) | BillingPlan Id..., Agent Id / Billed credit..., User Id / Credits used... | `Files/landing/studio/` (replaces the last file of the same kind) | `Consumption_Ingest_Studio` |
| Cowork credits (Viva Insights Consumption Dashboard) | ServiceId, MetricDate, Total Copilot Credits used... / SpendingPolicyId, PlanLimit... | `Files/landing/viva/` | `Consumption_Ingest_Viva` |
| Workday report | Primary Work Email | `Files/org_workday/workday.csv` (replaced) | `Copilot_Org_Data_Workday_Lander` |

**Where files go after a run**

- Files dropped straight into `Files/analytics_hub_uploads/` are moved to `_processed/<date>/` once routed,
  or to `_unrecognised/` when the header matches no export (or the source is set to Skip). The reason is in the log.
- Files under `Files/analytics_hub_uploads/sharepoint/` (an optional OneLake shortcut to a SharePoint folder)
  are read-only: they are copied, never moved, and the log stops the same file being loaded twice.
- Every decision is appended to the Delta table `dbo.analytics_hub_upload_log`.

The pipeline runs this before the loaders. A source with no file simply stays empty, so its page stays dormant.

In [ ]:
# === CONFIG ===
DROP_DIR        = 'Files/analytics_hub_uploads'             # the one drop folder
SHAREPOINT_DIR  = 'Files/analytics_hub_uploads/sharepoint'  # optional read-only shortcut to a SharePoint folder
LOG_TABLE       = 'dbo.analytics_hub_upload_log'            # one row per file per decision
ENABLED_SOURCES = ''                                        # comma-separated source ids; empty accepts every source
SIGNATURES_JSON = '[{"kind":"productFeedback","source":"productFeedback","groups":[["feedbackid"],["datesubmittedutc"],["feedbacktype"]],"dir":"Files/product_feedback","name":"feedback_{stamp}.csv","policy":"append"},{"kind":"agent365","source":"agent365","groups":[["agentname","name","displayname"],["titleid","packageid","agentcreatorid","creatorid","agenttypea365","publishertype","supportedin","availability"]],"dir":"Files/agent365","name":"agents.csv","policy":"replace"},{"kind":"studioTenant","source":"studioCredits","groups":[["billingplanid"],["environmentid"],["capacitytype"],["prepaidconsumedquantity"],["usagedate"]],"dir":"Files/landing/studio","name":"EntitlementConsumptionTenantDetailsReport_MCSMessages_{stamp}.csv","policy":"replaceKind"},{"kind":"studioAgent","source":"studioCredits","groups":[["agentid"],["agentname"],["billedcredit"],["nonbilledcredit"],["channel"]],"dir":"Files/landing/studio","name":"EntitlementConsumptionTenantPerAgentDetailsReport_MCSMessages_{stamp}.csv","policy":"replaceKind"},{"kind":"studioUser","source":"studioCredits","groups":[["userid"],["useremail"],["creditsused"],["billablecreditused"]],"dir":"Files/landing/studio","name":"EntitlementConsumptionTenantPerUserDetailsReport_MCSMessages_{stamp}.csv","policy":"replaceKind"},{"kind":"studioAgentDaily","source":"studioCredits","groups":[["usagedate"],["agentid"],["agentname"],["billedcredit"],["nonbilledcredit"],["channel"]],"dir":"Files/landing/studio","name":"StudioApiAgentDaily_{stamp}.csv","policy":"replaceKind"},{"kind":"studioUserDaily","source":"studioCredits","groups":[["usagedate"],["userid"],["billedcredit"],["nonbilledcredit"],["unit"]],"dir":"Files/landing/studio","name":"StudioApiUserDaily_{stamp}.csv","policy":"replaceKind"},{"kind":"studioEntitlement","source":"studioCredits","groups":[["snapshotdate"],["environmentid"],["environmentallocated"],["tenantprepaidconsumed"],["tenantpaygconsumed"]],"dir":"Files/landing/studio","name":"StudioApiEntitlement_{stamp}.csv","policy":"replaceKind"},{"kind":"vivaCredits","source":"coworkCredits","groups":[["serviceid"],["servicename"],["spendingpolicyid"],["metricdate"],["totalcopilotcreditsused"]],"dir":"Files/landing/viva","name":"PersonServiceCreditsMetrics_{stamp}.csv","policy":"append"},{"kind":"vivaPolicy","source":"coworkCredits","groups":[["spendingpolicyid"],["name"],["planlimit"],["userlimit"],["includedservices"]],"dir":"Files/landing/viva","name":"SpendingPolicyMetadata_{stamp}.csv","policy":"replaceKind"},{"kind":"workday","source":"workday","groups":[["primaryworkemail"]],"dir":"Files/org_workday","name":"workday.csv","policy":"replace"}]'

In [ ]:
# === HELPERS (pure, so they can be tested off Fabric) ===
import csv
import io
import json
import re
from datetime import datetime, timezone


def norm(name):
    # Lower case, letters and digits only: 'Feedback Id', 'feedback_id' and 'FeedbackID' all match.
    return re.sub(r'[^a-z0-9]', '', str(name).replace('\ufeff', '').lower())


def parse_header(text):
    text = (text or '').lstrip('\ufeff')
    if not text.strip():
        return []
    first = text.splitlines()[0]
    delimiter = max([',', ';', '\t'], key=first.count)
    try:
        row = next(csv.reader(io.StringIO(text), delimiter=delimiter))
    except StopIteration:
        return []
    return [c.strip() for c in row]


def enabled_set(text):
    ids = [s.strip() for s in str(text or '').split(',') if s.strip()]
    return set(ids) if ids else None


def detect(headers, signatures, enabled=None):
    # The signature matching the most column groups wins; a tie is ambiguous and nothing is loaded.
    have = {norm(h) for h in headers if norm(h)}
    if not have:
        return None, 'the file has no header row'
    hits = [s for s in signatures if all(any(h in have for h in g) for g in s['groups'])]
    if not hits:
        return None, 'the header matches no export Analytics Hub reads'
    top = max(len(s['groups']) for s in hits)
    best = [s for s in hits if len(s['groups']) == top]
    if len(best) > 1:
        return None, 'the header matches more than one export: ' + ', '.join(s['kind'] for s in best)
    sig = best[0]
    if enabled is not None and sig['source'] not in enabled:
        return None, f"it is a {sig['source']} export, but that source is set to Skip"
    return sig, ''


def kind_prefix(sig):
    # File-name prefix shared by every file of this kind, e.g. 'SpendingPolicyMetadata'.
    return sig['name'].split('{stamp}')[0].rstrip('_')


def target_path(sig, stamp):
    return f"{sig['dir']}/{sig['name'].replace('{stamp}', stamp)}"


def file_key(path, size, modified):
    return f'{path}|{size}|{modified}'

In [ ]:
# === ROUTE ===
import notebookutils
from pyspark.sql import types as T

SIGNATURES = json.loads(SIGNATURES_JSON)
ENABLED = enabled_set(ENABLED_SOURCES)
NOW = datetime.now(timezone.utc)
DAY = NOW.strftime('%Y-%m-%d')
RUN_STAMP = NOW.strftime('%Y%m%dT%H%M%SZ')


def _csvs(path):
    try:
        if not notebookutils.fs.exists(path):
            return []
        files = [f for f in notebookutils.fs.ls(path) if not f.isDir and f.name.lower().endswith('.csv')]
    except Exception as exc:
        print(f'cannot list {path}: {exc}')
        return []
    return sorted(files, key=lambda f: (getattr(f, 'modifyTime', 0) or 0, f.name))


def _place(src, dest, move):
    notebookutils.fs.mkdirs(dest.rsplit('/', 1)[0])
    if notebookutils.fs.exists(dest):
        notebookutils.fs.rm(dest)
    if move:
        notebookutils.fs.mv(src, dest)
    else:
        notebookutils.fs.cp(src, dest)


def _retire_kind(sig):
    # Moves earlier files of the same kind aside, so the loader reads only the newest export.
    prefix = kind_prefix(sig).lower()
    for f in _csvs(sig['dir']):
        if f.name.lower().startswith(prefix):
            _place(f.path, f"{sig['dir']}/_loaded/{DAY}/{f.name}", move=True)
            print(f'  retired {sig["dir"]}/{f.name}')


def _seen():
    if not spark.catalog.tableExists(LOG_TABLE):
        return set()
    rows = (spark.table(LOG_TABLE).where("origin = 'sharepoint' AND status <> 'failed'")
            .select('path', 'size', 'modified').collect())
    return {file_key(r['path'], r['size'], r['modified']) for r in rows}


LOG = []
SEEN = _seen()
counter = 0
for origin, folder in (('lakehouse', DROP_DIR), ('sharepoint', SHAREPOINT_DIR)):
    for f in _csvs(folder):
        modified = str(getattr(f, 'modifyTime', '') or '')
        rel = f'{folder}/{f.name}'
        if origin == 'sharepoint' and file_key(rel, f.size, modified) in SEEN:
            continue
        entry = {'run_at': NOW, 'file_name': f.name, 'path': rel, 'origin': origin, 'size': int(f.size or 0),
                 'modified': modified, 'source': None, 'kind': None, 'target': None, 'status': None, 'detail': ''}
        try:
            sig, reason = detect(parse_header(notebookutils.fs.head(f.path, 65536)), SIGNATURES, ENABLED)
            if sig is None:
                entry.update(status='unrecognised', detail=reason)
                if origin == 'lakehouse':
                    _place(f.path, f'{DROP_DIR}/_unrecognised/{f.name}', move=True)
                print(f'? {rel}: {reason}')
            else:
                counter += 1
                target = target_path(sig, f'{RUN_STAMP}_{counter:03d}')
                if sig['policy'] == 'replaceKind':
                    _retire_kind(sig)
                _place(f.path, target, move=False)
                if origin == 'lakehouse':
                    _place(f.path, f'{DROP_DIR}/_processed/{DAY}/{f.name}', move=True)
                entry.update(status='routed', source=sig['source'], kind=sig['kind'], target=target)
                print(f'> {rel} -> {target} ({sig["kind"]})')
        except Exception as exc:
            # Left where it is, so the next run tries again.
            entry.update(status='failed', detail=str(exc)[:1000])
            print(f'! {rel}: {exc}')
        LOG.append(entry)

SCHEMA = T.StructType([
    T.StructField('run_at', T.TimestampType()), T.StructField('file_name', T.StringType()),
    T.StructField('path', T.StringType()), T.StructField('origin', T.StringType()),
    T.StructField('size', T.LongType()), T.StructField('modified', T.StringType()),
    T.StructField('source', T.StringType()), T.StructField('kind', T.StringType()),
    T.StructField('target', T.StringType()), T.StructField('status', T.StringType()),
    T.StructField('detail', T.StringType()),
])
if LOG or not spark.catalog.tableExists(LOG_TABLE):
    (spark.createDataFrame([tuple(e[f.name] for f in SCHEMA.fields) for e in LOG], SCHEMA)
        .write.format('delta').mode('append').saveAsTable(LOG_TABLE))

summary = {status: sum(1 for e in LOG if e['status'] == status) for status in ('routed', 'unrecognised', 'failed')}
summary['sources'] = sorted({e['source'] for e in LOG if e['source']})
print(json.dumps(summary))
notebookutils.notebook.exit(json.dumps(summary))